# Phase 8: Data Augmentation and Transfer Learning (Q47–Q49)

**Questions covered:** Q47 (augmentation: before/after, overfitting), Q48 (MobileNetV2, ResNet50, VGG16, EfficientNet-B0: frozen vs fine-tuned), Q49 (validation-level evaluation and model comparison; the held-out **test** evaluation, error analysis and sample predictions are done once in Phase 9).

## Part A: augmentation (Q47)
We take the best custom CNN from Phase 7 and train it **twice with identical settings**, once without and once with augmentation. Random **rotation, horizontal flip, zoom, shift and shear** are applied on the GPU to each training batch. Validation images are never augmented.

## Part B: transfer learning (Q48)
Terminology (the PDF slide 5 and the DOCX use the words differently, so we fix them here and note it in the report):
- **Pretrained weights:** filters learned on ImageNet (1.28M images, 1,000 classes). ImageNet contains many cat and dog breeds, so these features should be very relevant.
- **Frozen backbone / feature extraction:** pretrained layers are not updated; only a new head is trained.
- **Fine-tuning:** the last pretrained blocks are *unfrozen* and trained with a small learning rate, together with the head.

For each model: load the ImageNet weights (from `models/pretrained/`, because the pod is offline), replace the 1,000-class classifier by a new head (global average pool → dropout → 1 logit), then

1. **Stage 1 (frozen):** because the frozen backbone never changes, its features are computed **once** and only the head is trained on them (fast). The resulting head is put into the full model and evaluated normally.
2. **Stage 2 (fine-tuned):** unfreeze the last blocks and train with small learning rates (head 1e-4, backbone 2e-5), early stopping on validation loss.

The frozen part is kept in `eval()` mode while fine-tuning, so its BatchNorm statistics are not disturbed.

Everything is judged on the **validation** set. **The test split is not loaded.** Not every bigger model is better: compare accuracy, size and speed.

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "8"
import sys, json, time, random, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path

SEED = 42
PROJECT_ROOT = Path("/workspace/sic_final_project")      # change here if the folder moves
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CNN_DIR, TL_DIR = PROJECT_ROOT / "models/cnn", PROJECT_ROOT / "models/transfer"
WEIGHTS_DIR = PROJECT_ROOT / "models/pretrained"
for d in [CNN_DIR, TL_DIR, PROJECT_ROOT / "results/figures", PROJECT_ROOT / "results/metrics", PROJECT_ROOT / "results/comparisons"]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- one switch for the whole notebook ----------
QUICK = True      # True: tiny epoch counts to test the pipeline (nothing is logged). Set False for the real run.
LOG = not QUICK
# --------------------------------------------------------

needed = ["data/processed/images_128.npy", "data/processed/images_224.npy", "data/processed/labels.npy",
          "data/splits/split_seed42.csv", "models/cnn/best_custom_cnn.json", "src/evaluation.py", "src/cnn_models.py", "src/training.py"]
missing = [f for f in needed if not (PROJECT_ROOT / f).exists()]
for f in needed:
    print("OK     " if f not in missing else "MISSING", f)
assert not missing, "Run Phase 7 with QUICK = False first (best_custom_cnn.json is created at its end)."
for n in ["mobilenet_v2", "resnet50", "vgg16", "efficientnet_b0"]:
    p = WEIGHTS_DIR / f"{n}_imagenet.pth"
    print("OK     " if p.exists() else "MISSING", p.name, f"{p.stat().st_size / 1e6:.0f} MB" if p.exists() else "")
assert torch.cuda.is_available(), "No GPU visible."
print("mode:", "QUICK (pipeline check, nothing logged)" if QUICK else "FULL run")

In [ ]:
%%writefile /workspace/sic_final_project/src/augmentation.py
"""GPU data augmentation for batches of 0-1 float images (N, 3, H, W): rotation, flip, zoom, shift and shear
are combined into ONE random affine transform per image and applied with a single resampling step."""
import math
import torch
import torch.nn.functional as F


def random_affine_batch(x, rot_deg=15.0, zoom=(0.85, 1.15), shift=0.10, shear_deg=10.0, flip=True):
    """rot_deg: max rotation (+-). zoom: factor range (>1 zooms in). shift: max shift as a fraction of the image size.
    shear_deg: max horizontal shear (+-). flip: random horizontal flip with p = 0.5. Border pixels are reflected."""
    B, dev = x.size(0), x.device
    u = lambda: torch.rand(B, device=dev) * 2 - 1                      # uniform in [-1, 1]
    ang = u() * math.radians(rot_deg)
    sh = u() * math.radians(shear_deg)
    z = torch.empty(B, device=dev).uniform_(zoom[0], zoom[1])
    tx, ty = u() * shift * 2, u() * shift * 2                           # normalised coords span [-1, 1]
    ones = torch.ones(B, device=dev)
    fl = torch.where(torch.rand(B, device=dev) < 0.5, -ones, ones) if flip else ones
    c, s, t = torch.cos(ang), torch.sin(ang), torch.tan(sh)
    # output->input mapping: rotation @ shear @ flip, divided by the zoom factor
    theta = torch.stack([torch.stack([c * fl / z, (c * t - s) / z, tx], dim=1),
                         torch.stack([s * fl / z, (s * t + c) / z, ty], dim=1)], dim=1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)


def make_augment_fn(**kwargs):
    return lambda x: random_affine_batch(x, **kwargs)

In [ ]:
%%writefile /workspace/sic_final_project/src/pretrained_models.py
"""ImageNet-pretrained backbones (torchvision) with a new binary head. Also used by the deployment package."""
import copy
import time
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

_CTORS = {"mobilenet_v2": models.mobilenet_v2, "resnet50": models.resnet50,
          "vgg16": models.vgg16, "efficientnet_b0": models.efficientnet_b0}
FEATURE_DIM = {"mobilenet_v2": 1280, "resnet50": 2048, "vgg16": 512, "efficientnet_b0": 1280}
# fine-tuning unfreezes backbone.children()[k:]  (last blocks)
UNFREEZE_FROM = {"mobilenet_v2": 14, "resnet50": 7, "vgg16": 24, "efficientnet_b0": 6}
PRETTY = {"mobilenet_v2": "MobileNetV2", "resnet50": "ResNet50", "vgg16": "VGG16", "efficientnet_b0": "EfficientNet-B0"}


def original_head(name):
    """The original 1000-class ImageNet classification head (for documentation)."""
    m = _CTORS[name](weights=None)
    return m.fc if name == "resnet50" else m.classifier


def build_backbone(name, weights_path=None):
    m = _CTORS[name](weights=None)
    if weights_path is not None:
        m.load_state_dict(torch.load(weights_path, map_location="cpu", weights_only=True))
    if name == "resnet50":
        return nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2, m.layer3, m.layer4)
    return m.features


class TransferModel(nn.Module):
    """backbone (pretrained) -> GAP -> Flatten -> Dropout -> Linear(feat, 1). Output: ONE logit (Dog = positive)."""

    def __init__(self, name, weights_path=None, dropout=0.3):
        super().__init__()
        self.backbone_name = name
        self.backbone = build_backbone(name, weights_path)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout),
                                  nn.Linear(FEATURE_DIM[name], 1))
        self.trainable_from = len(list(self.backbone.children()))       # default: whole backbone frozen
        self.set_trainable_from(self.trainable_from)

    def set_trainable_from(self, k):
        """Freeze backbone children [0, k) and unfreeze [k, end). The head is always trainable."""
        self.trainable_from = k
        for i, child in enumerate(self.backbone.children()):
            for p in child.parameters():
                p.requires_grad = i >= k
        self.train(self.training)

    def train(self, mode=True):
        super().train(mode)
        if mode:                                      # frozen blocks stay in eval(): BatchNorm statistics stay fixed
            for i, child in enumerate(self.backbone.children()):
                if i < self.trainable_from:
                    child.eval()
        return self

    def features(self, x):
        return self.head[:2](self.backbone(x))        # pooled feature vector, (N, feat)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


@torch.no_grad()
def extract_features(model, images_u8, mean, std, batch_size=128):
    """Pooled backbone features for a uint8 NHWC GPU tensor. Valid for a frozen backbone, with no augmentation."""
    from training import to_input, _stats
    dev = images_u8.device
    model.eval()
    mean_t, std_t = _stats(mean, std, dev)
    out = []
    for s in range(0, len(images_u8), batch_size):
        xb = to_input(images_u8[s:s + batch_size], mean_t, std_t)
        with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=dev.type == "cuda"):
            f = model.features(xb)
        out.append(f.float())
    return torch.cat(out)


def train_head_on_features(F_tr, y_tr, F_va, y_va, dropout=0.3, lr=1e-3, weight_decay=1e-4, epochs=60,
                           patience=8, batch_size=256, seed=42):
    """Train Dropout + Linear on cached features. Returns (head, history). Same history keys as training.train_model."""
    from training import seed_everything
    seed_everything(seed)
    dev = F_tr.device
    head = nn.Sequential(nn.Dropout(dropout), nn.Linear(F_tr.shape[1], 1)).to(dev)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc"]}
    best_loss, best_epoch, best_state, bad, t0 = float("inf"), 0, None, 0, time.perf_counter()
    for ep in range(1, epochs + 1):
        head.train()
        perm = torch.randperm(len(F_tr), device=dev, generator=gen)
        tot = torch.zeros((), device=dev); cor = torch.zeros((), device=dev)
        for i in range(0, len(F_tr), batch_size):
            idx = perm[i:i + batch_size]
            logits = head(F_tr[idx]).squeeze(1)
            loss = loss_fn(logits, y_tr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.detach() * len(idx); cor += ((logits.detach() > 0) == (y_tr[idx] > 0.5)).sum()
        head.eval()
        with torch.no_grad():
            lv = head(F_va).squeeze(1)
            va_loss = float(nn.functional.binary_cross_entropy_with_logits(lv, y_va))
            va_acc = float(((lv > 0) == (y_va > 0.5)).float().mean())
        for k, v in zip(hist.keys(), [float(tot) / len(F_tr), float(cor) / len(F_tr), va_loss, va_acc]):
            hist[k].append(v)
        if va_loss < best_loss - 1e-4:
            best_loss, best_epoch, best_state, bad = va_loss, ep, copy.deepcopy(head.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    head.load_state_dict(best_state)
    hist.update(best_epoch=best_epoch, best_val_loss=best_loss, epochs_run=len(hist["val_loss"]),
                train_time_s=time.perf_counter() - t0)
    return head, hist

# Part A: Data augmentation (Q47)

Load the 128×128 data and the best custom architecture from Phase 7 (`best_custom_cnn.json`). The same channel mean/std as in Phase 7 is reused, so the only difference between the two runs below is the augmentation.

In [ ]:
import evaluation, cnn_models, training, augmentation, pretrained_models
for m in (evaluation, cnn_models, training, augmentation, pretrained_models):
    importlib.reload(m)
dev = torch.device("cuda")
training.seed_everything(SEED)

best = json.loads((CNN_DIR / "best_custom_cnn.json").read_text())
print("Best custom architecture from Phase 7:", best["name"], "| val acc then:", round(best["val"]["accuracy"], 4))
mean128, std128 = np.array(best["mean"], np.float32), np.array(best["std"], np.float32)

split_df = pd.read_csv(PROJECT_ROOT / "data/splits/split_seed42.csv")
labels = np.load(PROJECT_ROOT / "data/processed/labels.npy")
tr = split_df.loc[split_df["split"] == "train", "array_index"].to_numpy()
va = split_df.loc[split_df["split"] == "val", "array_index"].to_numpy()
y_train, y_val = labels[tr], labels[va]
sub = np.random.default_rng(SEED).choice(len(tr), 3000, replace=False)
sub_t = torch.from_numpy(sub).to(dev)

imgs128 = np.load(PROJECT_ROOT / "data/processed/images_128.npy", mmap_mode="r")
Xtr = torch.from_numpy(np.ascontiguousarray(imgs128[tr])).to(dev)
Xva = torch.from_numpy(np.ascontiguousarray(imgs128[va])).to(dev)
ytr = torch.from_numpy(y_train).float().to(dev); yva = torch.from_numpy(y_val).float().to(dev)
print(f"train {tuple(Xtr.shape)} | val {tuple(Xva.shape)} | GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Q47: what each transform does, and a few random combined augmentations of the same image
base = Xtr[[5, 40]].permute(0, 3, 1, 2).float() / 255.0          # two training images, (2, 3, 128, 128) in 0-1
variants = [("original", None),
            ("rotation", dict(rot_deg=25, zoom=(1, 1), shift=0, shear_deg=0, flip=False)),
            ("h-flip", dict(rot_deg=0, zoom=(1, 1), shift=0, shear_deg=0, flip=True)),
            ("zoom", dict(rot_deg=0, zoom=(0.8, 1.3), shift=0, shear_deg=0, flip=False)),
            ("shift", dict(rot_deg=0, zoom=(1, 1), shift=0.2, shear_deg=0, flip=False)),
            ("shear", dict(rot_deg=0, zoom=(1, 1), shift=0, shear_deg=25, flip=False)),
            ("combined", dict())]
torch.manual_seed(1)
fig, ax = plt.subplots(2, len(variants), figsize=(2.3 * len(variants), 5))
for r in range(2):
    for c, (name, kw) in enumerate(variants):
        img = base[r:r + 1] if kw is None else augmentation.random_affine_batch(base[r:r + 1], **kw)
        ax[r, c].imshow(img[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()); ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(name)
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/08_augmentation_examples.png", dpi=110); plt.show()

### Controlled comparison: same architecture, same seed, same schedule

Only the augmentation differs. Both runs use the same maximum epochs and patience (augmentation usually needs more epochs before validation loss bottoms out).
Each is **one run (one seed)**: a difference of a few tenths of a point is within noise, so say so in the discussion.

In [ ]:
AUG_EPOCHS, AUG_PATIENCE = (3, 2) if QUICK else (40, 8)
BATCH, LR, WD = best["schedule"]["batch"], best["schedule"]["lr"], best["schedule"]["weight_decay"]
RUN_CONTROL = True            # retrain the no-augmentation control with the SAME budget as the augmented run
aug_fn = augmentation.make_augment_fn(rot_deg=15, zoom=(0.85, 1.15), shift=0.10, shear_deg=10, flip=True)
aug_runs = {}

def run_cnn(tag, label, augment_fn):
    model = cnn_models.build_custom_cnn(best["arch_kwargs"], input_size=best["input_size"])
    ckpt = CNN_DIR / f"{tag}{'_quick' if QUICK else ''}.pt"
    print(f"\n=== {label} ({best['name']}) ===")
    hist = training.train_model(model, Xtr, ytr, Xva, yva, mean128, std128, epochs=AUG_EPOCHS, batch_size=BATCH, lr=LR,
                                weight_decay=WD, patience=AUG_PATIENCE, augment_fn=augment_fn, seed=SEED, ckpt_path=ckpt)
    metrics, pred, probs, ms = training.evaluate_dl(model, Xva, y_val, mean128, std128)
    p_tr = training.predict_probs(model, Xtr[sub_t], mean128, std128)
    train_acc = float(((p_tr > 0.5).astype(int) == y_train[sub]).mean())
    total, trainable = cnn_models.count_params(model)
    if LOG:
        evaluation.log_result(label, "val", metrics, train_accuracy=train_acc, train_time_s=round(hist["train_time_s"], 1),
                              infer_ms_per_img=round(ms, 4), params=total, trainable_params=trainable,
                              epochs_run=hist["epochs_run"], best_epoch=hist["best_epoch"],
                              notes=f"{best['name']} | augmentation: {'rotation, flip, zoom, shift, shear' if augment_fn else 'none'} | GPU")
    aug_runs[label] = dict(hist=hist, metrics=metrics, pred=pred, probs=probs, train_acc=train_acc, params=total, ms=ms, ckpt=str(ckpt))
    print(f"-> val acc {metrics['accuracy']:.4f} | AUC {metrics['roc_auc']:.4f} | train acc (eval mode) {train_acc:.4f} | "
          f"best epoch {hist['best_epoch']}/{hist['epochs_run']} | {hist['train_time_s']:.0f}s")
    del model; torch.cuda.empty_cache()

if RUN_CONTROL:
    run_cnn("cnn_control_noaug", "Custom CNN (control, no aug)", None)
run_cnn("cnn_aug", "CNN + Augmentation", aug_fn)

In [ ]:
fig, axes = plt.subplots(len(aug_runs), 2, figsize=(11, 3.5 * len(aug_runs)))
axes = np.atleast_2d(axes)
for row, (label, r) in zip(axes, aug_runs.items()):
    training.plot_history(r["hist"], title=label, axes=row)
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/08_augmentation_curves.png", dpi=110); plt.show()

rows = []
for label, r in aug_runs.items():
    m, h = r["metrics"], r["hist"]
    rows.append({"run": label, "val acc": m["accuracy"], "val F1": m["f1"], "val AUC": m["roc_auc"],
                 "best val loss": h["best_val_loss"], "train acc (eval mode)": r["train_acc"],
                 "train-val gap": r["train_acc"] - m["accuracy"], "best epoch": h["best_epoch"], "epochs run": h["epochs_run"],
                 "train time s": h["train_time_s"]})
q47 = pd.DataFrame(rows).set_index("run"); display(q47.round(4))
q47.to_csv(PROJECT_ROOT / "results/comparisons/08_augmentation_val.csv")

In [ ]:
# free the 128-pixel data before loading the 224-pixel set
del Xtr, Xva, ytr, yva, sub_t
torch.cuda.empty_cache()
print("GPU memory after cleanup:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

# Part B: Transfer learning (Q48)

224×224 images (the ImageNet training size) normalised with the **ImageNet** mean/std, as the pretrained weights expect. The training set as uint8 is about 2.6 GB on the GPU.

In [ ]:
imgs224 = np.load(PROJECT_ROOT / "data/processed/images_224.npy", mmap_mode="r")
mean224 = np.array([0.485, 0.456, 0.406], np.float32); std224 = np.array([0.229, 0.224, 0.225], np.float32)
Xtr = torch.from_numpy(np.ascontiguousarray(imgs224[tr])).to(dev)
Xva = torch.from_numpy(np.ascontiguousarray(imgs224[va])).to(dev)
ytr = torch.from_numpy(y_train).float().to(dev); yva = torch.from_numpy(y_val).float().to(dev)
sub_t = torch.from_numpy(sub).to(dev)
print(f"train {tuple(Xtr.shape)} | val {tuple(Xva.shape)} | GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Understand the original architectures (Q48)
rows = []
for n in ["mobilenet_v2", "resnet50", "vgg16", "efficientnet_b0"]:
    m = pretrained_models.TransferModel(n, weights_path=None)
    total, _ = cnn_models.count_params(m)
    k = pretrained_models.UNFREEZE_FROM[n]
    m.set_trainable_from(k)
    rows.append({"model": pretrained_models.PRETTY[n], "backbone children": len(list(m.backbone.children())),
                 "feature dim": pretrained_models.FEATURE_DIM[n], "total params (M)": round(total / 1e6, 2),
                 "unfrozen from block": k, "trainable when fine-tuning (M)": round(cnn_models.count_params(m)[1] / 1e6, 2),
                 "original head": str(pretrained_models.original_head(n)).replace("\n", " ")[:90]})
    del m
display(pd.DataFrame(rows))

In [ ]:
# ---------- settings ----------
HEAD_EPOCHS = 5 if QUICK else 60
FT_EPOCHS, FT_PATIENCE, FT_BATCH = (1, 1, 64) if QUICK else (8, 3, 64)
FT_LR_HEAD, FT_LR_BACKBONE = 1e-4, 2e-5
# ------------------------------
TL_RESULTS = {}

def run_transfer(name):
    pretty = pretrained_models.PRETTY[name]
    model = pretrained_models.TransferModel(name, weights_path=WEIGHTS_DIR / f"{name}_imagenet.pth").to(dev)
    model = model.to(memory_format=torch.channels_last)
    total, _ = cnn_models.count_params(model)
    n_blocks = len(list(model.backbone.children()))
    print(f"\n=== {pretty}: {total / 1e6:.1f}M parameters ===")

    # ---- Stage 1: frozen backbone. Features are computed once, only the head is trained ----
    model.set_trainable_from(n_blocks)
    t0 = time.perf_counter()
    F_tr = pretrained_models.extract_features(model, Xtr, mean224, std224); F_va = pretrained_models.extract_features(model, Xva, mean224, std224)
    torch.cuda.synchronize(); t_feat = time.perf_counter() - t0
    head, h1 = pretrained_models.train_head_on_features(F_tr, ytr, F_va, yva, epochs=HEAD_EPOCHS, seed=SEED)
    model.head[3].load_state_dict(head[1].state_dict())
    m1, pred1, probs1, ms1 = training.evaluate_dl(model, Xva, y_val, mean224, std224, batch_size=128)
    p_tr = training.predict_probs(model, Xtr[sub_t], mean224, std224, batch_size=128)
    tr1 = float(((p_tr > 0.5).astype(int) == y_train[sub]).mean())
    print(f"[frozen]     feature extraction {t_feat:.0f}s + head {h1['train_time_s']:.0f}s | val acc {m1['accuracy']:.4f} "
          f"(cached-feature head said {h1['val_acc'][h1['best_epoch'] - 1]:.4f}) | AUC {m1['roc_auc']:.4f} | train acc {tr1:.4f}")
    torch.save(head[1].state_dict(), TL_DIR / f"{name}_head_frozen{'_quick' if QUICK else ''}.pt")
    if LOG:
        evaluation.log_result(f"TL {pretty} (frozen)", "val", m1, train_accuracy=tr1, train_time_s=round(t_feat + h1["train_time_s"], 1),
                              infer_ms_per_img=round(ms1, 4), params=total, trainable_params=FEATURE_DIM_HEAD(name),
                              epochs_run=h1["epochs_run"], best_epoch=h1["best_epoch"],
                              notes="ImageNet backbone frozen, new head trained on cached features | GPU")
    del F_tr, F_va; torch.cuda.empty_cache()

    # ---- Stage 2: fine-tune the last blocks ----
    k = pretrained_models.UNFREEZE_FROM[name]
    model.set_trainable_from(k)
    head_params = list(model.head.parameters())
    bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
    n_train = sum(p.numel() for p in head_params + bb_params)
    ckpt = TL_DIR / f"{name}_finetuned{'_quick' if QUICK else ''}.pt"
    h2 = training.train_model(model, Xtr, ytr, Xva, yva, mean224, std224, epochs=FT_EPOCHS, batch_size=FT_BATCH,
                              patience=FT_PATIENCE, lr=FT_LR_HEAD, seed=SEED, ckpt_path=ckpt, verbose=True,
                              param_groups=[{"params": head_params, "lr": FT_LR_HEAD}, {"params": bb_params, "lr": FT_LR_BACKBONE}])
    m2, pred2, probs2, ms2 = training.evaluate_dl(model, Xva, y_val, mean224, std224, batch_size=128)
    p_tr = training.predict_probs(model, Xtr[sub_t], mean224, std224, batch_size=128)
    tr2 = float(((p_tr > 0.5).astype(int) == y_train[sub]).mean())
    print(f"[fine-tuned] {h2['train_time_s']:.0f}s | val acc {m2['accuracy']:.4f} | AUC {m2['roc_auc']:.4f} | train acc {tr2:.4f} | "
          f"best epoch {h2['best_epoch']}/{h2['epochs_run']} | trainable {n_train / 1e6:.2f}M")
    if LOG:
        evaluation.log_result(f"TL {pretty} (fine-tuned)", "val", m2, train_accuracy=tr2, train_time_s=round(t_feat + h1["train_time_s"] + h2["train_time_s"], 1),
                              infer_ms_per_img=round(ms2, 4), params=total, trainable_params=n_train,
                              epochs_run=h2["epochs_run"], best_epoch=h2["best_epoch"],
                              notes=f"unfroze backbone blocks {k}+, lr head {FT_LR_HEAD}, backbone {FT_LR_BACKBONE} | GPU")
    TL_RESULTS[name] = {"frozen": dict(hist=h1, metrics=m1, probs=probs1, pred=pred1, train_acc=tr1, ms=ms1),
                        "finetuned": dict(hist=h2, metrics=m2, probs=probs2, pred=pred2, train_acc=tr2, ms=ms2,
                                          ckpt=str(ckpt), trainable=n_train),
                        "params": total}
    del model, head; torch.cuda.empty_cache()

def FEATURE_DIM_HEAD(name):
    return pretrained_models.FEATURE_DIM[name] + 1          # Linear(feat, 1): weights + bias

run_transfer("mobilenet_v2")        # smallest model first: if something is wrong, you find out quickly

In [ ]:
run_transfer("resnet50")

In [ ]:
run_transfer("vgg16")                # the largest model: ~138M parameters. If it runs out of GPU memory, set FT_BATCH = 32 above

In [ ]:
run_transfer("efficientnet_b0")      # optional extra experiment

## Frozen vs fine-tuned, and model comparison (validation set)

In [ ]:
rows = []
for n, r in TL_RESULTS.items():
    for stage, key in [("frozen", "frozen"), ("fine-tuned", "finetuned")]:
        m = r[key]["metrics"]
        rows.append({"model": pretrained_models.PRETTY[n], "stage": stage, "val acc": m["accuracy"], "val F1": m["f1"],
                     "val AUC": m["roc_auc"], "train acc": r[key]["train_acc"], "gap": r[key]["train_acc"] - m["accuracy"],
                     "params (M)": r["params"] / 1e6, "size MB (fp32)": r["params"] * 4 / 1e6,
                     "ms/img": r[key]["ms"], "best epoch": r[key]["hist"]["best_epoch"]})
tl = pd.DataFrame(rows)
display(tl.round(4))
tl.to_csv(PROJECT_ROOT / "results/comparisons/08_transfer_val.csv", index=False)

piv = tl.pivot(index="model", columns="stage", values="val acc")[["frozen", "fine-tuned"]]
ax = piv.plot.bar(figsize=(8, 4), rot=0); ax.set_ylim(max(0.5, piv.values.min() - 0.03), min(1.0, piv.values.max() + 0.01))
ax.set_title("Transfer learning: frozen vs fine-tuned (validation accuracy)"); ax.set_ylabel("accuracy")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/08_transfer_frozen_vs_ft.png", dpi=110); plt.show()

In [ ]:
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for n, r in TL_RESULTS.items():
    fpr, tpr, _ = roc_curve(y_val, r["finetuned"]["probs"]); ax[0].plot(fpr, tpr, label=f"{pretrained_models.PRETTY[n]} (AUC {r['finetuned']['metrics']['roc_auc']:.4f})")
for label, r in aug_runs.items():
    fpr, tpr, _ = roc_curve(y_val, r["probs"]); ax[0].plot(fpr, tpr, ls="--", label=f"{label} (AUC {r['metrics']['roc_auc']:.4f})")
ax[0].plot([0, 1], [0, 1], "k:", lw=1); ax[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC (validation), fine-tuned models")
ax[0].legend(fontsize=7)
for n, r in TL_RESULTS.items():                      # zoom into the top-left corner, where strong models differ
    fpr, tpr, _ = roc_curve(y_val, r["finetuned"]["probs"]); ax[1].plot(fpr, tpr, label=pretrained_models.PRETTY[n])
ax[1].set(xlim=(0, 0.1), ylim=(0.9, 1.0), xlabel="false positive rate", ylabel="true positive rate", title="zoom: top-left corner")
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/08_roc_val.png", dpi=110); plt.show()

cols = len(TL_RESULTS)
fig, axes = plt.subplots(1, cols, figsize=(3.8 * cols, 3.6))
for ax_, (n, r) in zip(np.atleast_1d(axes), TL_RESULTS.items()):
    evaluation.plot_confusion(y_val, r["finetuned"]["pred"], f"{pretrained_models.PRETTY[n]} (fine-tuned)", ax=ax_)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(len(TL_RESULTS), 4, figsize=(18, 3.4 * len(TL_RESULTS)))
axes = np.atleast_2d(axes)
for row, (n, r) in zip(axes, TL_RESULTS.items()):
    training.plot_history(r["frozen"]["hist"], title=f"{pretrained_models.PRETTY[n]} frozen", axes=row[:2])
    training.plot_history(r["finetuned"]["hist"], title=f"{pretrained_models.PRETTY[n]} fine-tuned", axes=row[2:])
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/08_transfer_curves.png", dpi=100); plt.show()

## Best transfer-learning model on validation (Q49, validation level)

Rule: highest validation accuracy; if within 0.2 percentage points, prefer the model with **fewer parameters**. This is only the best *transfer* candidate. The **final** model is chosen in Phase 9 (still on validation data) by also weighing the custom CNNs, size and speed. Skip the saving part while `QUICK = True`.

In [ ]:
from sklearn.metrics import classification_report
if QUICK:
    print("QUICK run: nothing is saved as 'best'. Set QUICK = False at the top, rerun the notebook, then run this cell.")
else:
    ranked = tl.sort_values("val acc", ascending=False)
    near = ranked[ranked["val acc"] >= ranked["val acc"].iloc[0] - 0.002]
    pick = near.sort_values("params (M)").iloc[0]
    bb = [k for k, v in pretrained_models.PRETTY.items() if v == pick["model"]][0]
    stage = "finetuned" if pick["stage"] == "fine-tuned" else "frozen"
    print("Best transfer-learning candidate:", pick["model"], f"({pick['stage']})  val acc {pick['val acc']:.4f}")
    info = {"backbone": bb, "pretty": pick["model"], "stage": stage, "dropout": 0.3, "input_size": 224,
            "mean": [float(v) for v in mean224], "std": [float(v) for v in std224],
            "ckpt": f"models/transfer/{bb}_finetuned.pt" if stage == "finetuned" else f"models/transfer/{bb}_head_frozen.pt",
            "val": {k: float(v) for k, v in TL_RESULTS[bb][stage]["metrics"].items()}}
    (TL_DIR / "best_transfer.json").write_text(json.dumps(info, indent=2))
    print("saved models/transfer/best_transfer.json")
    print(classification_report(y_val, TL_RESULTS[bb][stage]["pred"], target_names=evaluation.CLASS_NAMES, digits=4))
    for p in sorted(TL_DIR.iterdir()):
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.name}")

## Your write-ups (use your own numbers and words)

**Q47.** Show the augmentation examples and explain what each transform simulates (camera angle, mirrored pose, distance, framing). Compare the control and augmented runs: validation accuracy, F1, AUC, **train-vs-validation gap**, best epoch. Did augmentation reduce overfitting? Be honest that this is one seed, so a small difference may be noise.

**Q48.** For each of MobileNetV2, ResNet50, VGG16 (and EfficientNet-B0): describe the original architecture (the table above), the new head, what was frozen, what was fine-tuned, and the learning rates. Compare frozen vs fine-tuned using the table and bar chart. Does the bigger model win? Weigh accuracy against parameters, size and speed. Explain pretrained weights and why ImageNet features suit cats and dogs.

**Q49 (validation level).** Quote the best candidate's accuracy, precision, recall, F1 and AUC (classification report), and describe its confusion matrix. The test-set numbers, error analysis and sample predictions come in Phase 9.

**For the report:** the frozen stage trains only the head on cached features (the backbone is deterministic without augmentation); fine-tuning unfreezes the last blocks with learning rates 1e-4 (head) and 2e-5 (backbone); ImageNet normalisation; bfloat16 mixed precision; one seed per run; hyperparameters chosen on validation; test set untouched. **PDF slide 5 calls the frozen setting "fine-tuning"; this project uses the DOCX/standard terminology.**

Files created: `src/augmentation.py`, `src/pretrained_models.py`, `models/cnn/cnn_aug.pt`, `models/cnn/cnn_control_noaug.pt`, `models/transfer/*.pt`, `models/transfer/best_transfer.json`, `results/comparisons/08_*.csv`, figures `08_*.png`, and rows in `results/comparisons/model_results.csv`.